from pysparq\.sql import functions as F

In [0]:
from pyspark.sql import functions as F


In [0]:
df_silver = spark.read.table('weather.silver.silver_weather')
display(df_silver)

In [0]:
df_duplicates = df_silver.groupBy("time").count().filter("count > 1")
display(df_duplicates)

In [0]:
# Daily Summary gold table 

df_gold_daily = df_silver.groupBy("year", "month", "date").agg(
    F.round(F.avg("temperature_2m"), 2).alias("avg_temp"),
    F.max("temperature_2m").alias("max_temp"),
    F.min("temperature_2m").alias("min_temp"),
    F.round(F.sum("rain"), 2).alias("total_rain"),
    F.round(F.sum("snowfall"), 2).alias("total_snowfall"),
    F.round(F.avg("wind_speed_10m"), 2).alias("avg_wind_speed_10m"),
    F.max('wind_speed_100m').alias("max_wind_100m")
)

In [0]:
# Monthly Summary gold table 

df_gold_monthly = df_silver.groupBy("year", "month").agg(
    F.round(F.avg("temperature_2m"), 2).alias("avg_temp"),
    F.max("temperature_2m").alias("max_temp"),
    F.min("temperature_2m").alias("min_temp"),
    F.round(F.sum("rain"), 2).alias("total_rain"),
    F.round(F.sum("snowfall"), 2).alias("total_snowfall"),
    # F.max("wind_speed_10m_kmh").alias("max_wind_gust_kmh")
    F.round(F.avg("wind_speed_10m"), 2).alias("avg_wind_speed_10m")
)


In [0]:
# Annual Summary gold table

df_gold_annual = df_silver.groupBy("year").agg(
    F.round(F.avg("temperature_2m"), 2).alias("avg_temp"),
    F.max("temperature_2m").alias("max_temp"),
    F.min("temperature_2m").alias("min_temp"),
    F.round(F.sum("rain"), 2).alias("total_rain"),
    F.round(F.sum("snowfall"), 2).alias("total_snowfall"),
    # F.max("wind_speed_10m_kmh").alias("max_wind_gust_kmh")
    F.round(F.avg("wind_speed_10m"), 2).alias("avg_wind_speed_10m")
)


In [0]:
(df_gold_daily.write
    .format("delta")
    .mode("overwrite")
    .partitionBy("year")
    .saveAsTable("weather.gold.gold_daily_summary"))

(df_gold_monthly.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("weather.gold.gold_monthly_summary"))

(df_gold_annual.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("weather.gold.gold_annual_summary"))

In [0]:
display(df_gold_annual)